In [4]:
import numpy as np
import random
from typing import List, Dict
from datetime import datetime

# Transaction Logging Class
class TradingTransactionLog:
    def __init__(self, network_id: str):
        """
        Initialize a transaction log for a specific neural network
        
        Args:
        - network_id: Unique identifier for the neural network
        """
        self.network_id = network_id
        self.transactions = []
    
    def log_transaction(
        self, 
        action: str, 
        stock_price: float, 
        quantity: float, 
        cash_before: float, 
        cash_after: float, 
        stocks_before: float, 
        stocks_after: float,
        fundamental_score: float,
        technical_score: float,
        sentiment_score: float
    ):
        """
        Log a detailed transaction
        
        Args:
        - Various transaction and market state parameters
        """
        transaction = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "network_id": self.network_id,
            "action": action,
            "stock_price": stock_price,
            "quantity": quantity,
            "cash_before": cash_before,
            "cash_after": cash_after,
            "stocks_before": stocks_before,
            "stocks_after": stocks_after,
            "fundamental_score": fundamental_score,
            "technical_score": technical_score,
            "sentiment_score": sentiment_score
        }
        self.transactions.append(transaction)
    
    def print_log(self):
        """
        Print out the entire transaction log
        """
        print(f"\nTransaction Log for Network {self.network_id}:")
        for transaction in self.transactions:
            print(f"{transaction['timestamp']} | Action: {transaction['action']} | "
                  f"Stock Price: ${transaction['stock_price']:.2f} | "
                  f"Quantity: {transaction['quantity']:.2f} | "
                  f"Cash (Before/After): ${transaction['cash_before']:.2f}/${transaction['cash_after']:.2f} | "
                  f"Stocks (Before/After): {transaction['stocks_before']:.2f}/{transaction['stocks_after']:.2f}")
    
    def get_performance_summary(self):
        """
        Generate a performance summary of the transactions
        
        Returns:
        - Dictionary with performance metrics
        """
        if not self.transactions:
            return {}
        
        buy_transactions = [t for t in self.transactions if t['action'] == 'buy']
        sell_transactions = [t for t in self.transactions if t['action'] == 'sell']
        
        return {
            "total_transactions": len(self.transactions),
            "buy_transactions": len(buy_transactions),
            "sell_transactions": len(sell_transactions),
            "total_stocks_traded": sum(abs(t['quantity']) for t in self.transactions),
            "first_transaction": self.transactions[0]['timestamp'],
            "last_transaction": self.transactions[-1]['timestamp']
        }

# Modify TradingNeuralNetwork to include transaction logging
class TradingNeuralNetwork:
    def __init__(self, input_size: int, hidden_layers: List[int], output_size: int, network_id: str = None):
        """
        Initialize a neural network for trading decisions with transaction logging
        
        Args:
        - input_size: Number of input features
        - hidden_layers: List of neurons in each hidden layer
        - output_size: Number of output neurons (actions and quantities)
        - network_id: Unique identifier for the network
        """
        # Generate a unique network ID if not provided
        self.network_id = network_id or f"network_{random.randint(1000, 9999)}"
        
        # Create transaction log
        self.transaction_log = TradingTransactionLog(self.network_id)
        
        # Rest of the original initialization remains the same
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.output_size = output_size
        
        # Initialize network weights and biases (same as before)
        self.weights = []
        self.biases = []
        
        # Input to first hidden layer
        self.weights.append(np.random.randn(input_size, hidden_layers[0]) * 0.01)
        self.biases.append(np.zeros((1, hidden_layers[0])))
        
        # Hidden layers
        for i in range(1, len(hidden_layers)):
            self.weights.append(np.random.randn(hidden_layers[i-1], hidden_layers[i]) * 0.01)
            self.biases.append(np.zeros((1, hidden_layers[i])))
        
        # Output layer
        self.weights.append(np.random.randn(hidden_layers[-1], output_size) * 0.01)
        self.biases.append(np.zeros((1, output_size)))
        
        # Fitness score
        self.fitness = 0

    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """
        Perform a forward pass through the neural network.
        
        Args:
        - inputs: A numpy array of input features
        
        Returns:
        - outputs: A numpy array representing the network's output
        """
        activation = inputs
        for weight, bias in zip(self.weights, self.biases):
            # Linear transformation: z = W * a + b
            z = np.dot(activation, weight) + bias
            # Apply activation function (ReLU for hidden layers, softmax for output)
            activation = self.relu(z) if weight.shape[1] != self.output_size else self.softmax(z)
        return activation
    
    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        """
        Apply the ReLU activation function element-wise.
        
        Args:
        - x: Input array
        
        Returns:
        - Result after applying ReLU
        """
        return np.maximum(0, x)

    @staticmethod
    def softmax(x: np.ndarray) -> np.ndarray:
        """
        Apply the softmax activation function element-wise.
        
        Args:
        - x: Input array
        
        Returns:
        - Result after applying softmax
        """
        exp_x = np.exp(x - np.max(x))  # Stability improvement
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)


    
    def interpret_output(self, output: np.ndarray) -> Dict[str, float]:
        """
        Interpret the neural network's output to determine an action and its magnitude.
        
        Args:
        - output: A numpy array from the forward pass (network output)
        
        Returns:
        - Dictionary with the action and quantity percentage
        """
        # Split output into action probabilities and quantity probabilities
        action_probs = output[0, :3]  # First 3 outputs: buy, sell, hold
        quantity_probs = output[0, 3:]  # Next 3 outputs: quantity percentages

        # Determine the primary action
        action_index = np.argmax(action_probs)  # Index of the highest action probability
        actions = ['buy', 'sell', 'hold']
        primary_action = actions[action_index]

        # Determine the quantity percentage
        quantity_percentage = np.max(quantity_probs)  # Use the max quantity percentage

        return {
            'action': primary_action,
            'quantity_percentage': quantity_percentage
        }
        

# Modify GeneticTradingAlgorithm to use enhanced logging
class GeneticTradingAlgorithm:
    def __init__(
        self, 
        input_size: int, 
        hidden_layers: List[int], 
        output_size: int, 
        population_size: int = 100
    ):
        """
        Initialize Genetic Algorithm for Trading Neural Networks
        
        Args:
        - Same as before, with added transaction logging
        """
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.output_size = output_size
        self.population_size = population_size
        
        # Initialize population with unique network IDs
        self.population = [
            TradingNeuralNetwork(
                input_size, 
                hidden_layers, 
                output_size, 
                network_id=f"network_{i}"
            ) for i in range(population_size)
        ]
    
    def evaluate_fitness(
        self, 
        networks: List[TradingNeuralNetwork], 
        historical_data: List[Dict[str, float]]
    ) -> List[TradingNeuralNetwork]:
        """
        Evaluate fitness of neural networks with enhanced logging
        
        Args:
        - networks: List of neural networks to evaluate
        - historical_data: List of historical trading data points
        
        Returns:
        - Networks with updated fitness scores and transaction logs
        """
        for network in networks:
            total_portfolio_value = 1000  # Starting portfolio value
            current_cash = total_portfolio_value
            current_stocks = 0
            
            for data_point in historical_data:
                # Prepare input features
                input_features = np.array([
                    data_point['stock_price'],
                    data_point['fundamental_score'],
                    data_point['technical_score'],
                    data_point['sentiment_score'],
                    current_cash,
                    current_stocks
                ])
                
                # Get network decision
                output = network.forward(input_features)
                decision = network.interpret_output(output)
                
                # Simulate trading with detailed logging
                stock_price = data_point['stock_price']
                
                if decision['action'] == 'buy' and current_cash > 0:
                    # Calculate buy amount and stocks
                    buy_amount = current_cash * decision['quantity_percentage']
                    stocks_bought = buy_amount / stock_price
                    
                    # Log the buy transaction
                    network.transaction_log.log_transaction(
                        action='buy',
                        stock_price=stock_price,
                        quantity=stocks_bought,
                        cash_before=current_cash,
                        cash_after=current_cash - buy_amount,
                        stocks_before=current_stocks,
                        stocks_after=current_stocks + stocks_bought,
                        fundamental_score=data_point['fundamental_score'],
                        technical_score=data_point['technical_score'],
                        sentiment_score=data_point['sentiment_score']
                    )
                    
                    # Update portfolio
                    current_stocks += stocks_bought
                    current_cash -= buy_amount
                
                elif decision['action'] == 'sell' and current_stocks > 0:
                    # Calculate sell amount and stocks
                    sell_amount = current_stocks * decision['quantity_percentage']
                    
                    # Log the sell transaction
                    network.transaction_log.log_transaction(
                        action='sell',
                        stock_price=stock_price,
                        quantity=sell_amount,
                        cash_before=current_cash,
                        cash_after=current_cash + (sell_amount * stock_price),
                        stocks_before=current_stocks,
                        stocks_after=current_stocks - sell_amount,
                        fundamental_score=data_point['fundamental_score'],
                        technical_score=data_point['technical_score'],
                        sentiment_score=data_point['sentiment_score']
                    )
                    
                    # Update portfolio
                    current_cash += sell_amount * stock_price
                    current_stocks -= sell_amount
                
                else:
                    # Log do-nothing transaction
                    network.transaction_log.log_transaction(
                        action='hold',
                        stock_price=stock_price,
                        quantity=0,
                        cash_before=current_cash,
                        cash_after=current_cash,
                        stocks_before=current_stocks,
                        stocks_after=current_stocks,
                        fundamental_score=data_point['fundamental_score'],
                        technical_score=data_point['technical_score'],
                        sentiment_score=data_point['sentiment_score']
                    )
            
            # Calculate final portfolio value
            final_portfolio_value = current_cash + (current_stocks * stock_price)
            network.fitness = final_portfolio_value - total_portfolio_value
        
        return networks
        
    def mutate(self, network: TradingNeuralNetwork, mutation_rate: float = 0.1) -> TradingNeuralNetwork:
        """
        Mutate network weights
        
        Args:
        - network: Neural network to mutate
        - mutation_rate: Probability of mutation
        
        Returns:
        - Mutated neural network
        """
        mutated_network = network
        for i in range(len(mutated_network.weights)):
            # Randomly mutate weights
            mask = np.random.random(mutated_network.weights[i].shape) < mutation_rate
            mutations = np.random.randn(*mutated_network.weights[i].shape) * 0.1
            mutated_network.weights[i][mask] += mutations[mask]
        
        return mutated_network
    
    def crossover(self, parent1: TradingNeuralNetwork, parent2: TradingNeuralNetwork) -> TradingNeuralNetwork:
        """
        Perform crossover between two parent networks
        
        Args:
        - parent1, parent2: Parent neural networks
        
        Returns:
        - Child neural network
        """
        child = TradingNeuralNetwork(self.input_size, self.hidden_layers, self.output_size)
        
        # Randomly choose weights from each parent
        for i in range(len(child.weights)):
            mask = np.random.random(child.weights[i].shape) < 0.5
            child.weights[i] = np.where(mask, parent1.weights[i], parent2.weights[i])
        
        return child
    
    def evolve_generation(
        self, 
        historical_data: List[Dict[str, float]], 
        generations: int = 50, 
        elite_size: int = 10,
        log_top_performers: int = 3
    ) -> TradingNeuralNetwork:
        """
        Evolve multiple generations of trading neural networks with detailed logging
        
        Args:
        - Same as before, with added logging of top performers
        
        Returns:
        - Best performing neural network
        """
        for generation in range(generations):
            # Evaluate fitness of current population
            self.population = self.evaluate_fitness(self.population, historical_data)
            
            # Sort population by fitness (descending)
            self.population.sort(key=lambda x: x.fitness, reverse=True)
            
            # Print generation stats
            print(f"\nGeneration {generation + 1}:")
            print(f"Best Fitness = {self.population[0].fitness}")
            print(f"Average Fitness = {np.mean([n.fitness for n in self.population])}")
            
            # Log top performing networks' transactions
            print("\nTop Performers Transaction Summaries:")
            for i, network in enumerate(self.population[:log_top_performers], 1):
                print(f"\nTop Network {i} (ID: {network.network_id}):")
                network.transaction_log.print_log()
                
                # Print performance summary
                performance_summary = network.transaction_log.get_performance_summary()
                print("\nPerformance Summary:")
                for key, value in performance_summary.items():
                    print(f"{key}: {value}")
            
            elite_networks = self.population[:elite_size]
            
            # Create new generation
            new_population = elite_networks.copy()
            
            while len(new_population) < self.population_size:
                # Select parents (tournament selection)
                parent1 = max(random.sample(self.population, 5), key=lambda x: x.fitness)
                parent2 = max(random.sample(self.population, 5), key=lambda x: x.fitness)
                
                # Create child through crossover and mutation
                child = self.crossover(parent1, parent2)
                child = self.mutate(child)
                
                new_population.append(child)
            
            self.population = new_population
            
            # Print generation stats
            print(f"Generation {generation + 1}: "
                  f"Best Fitness = {self.population[0].fitness}, "
                  f"Average Fitness = {np.mean([n.fitness for n in self.population])}")
        
        return self.population[0] 
            
            # Rest of the evolution process remains the same
            # (Select elite networks, create new generation, etc.)
            
            # ... (rest of the original method)
        
       

# Rest of the code remains the same as in the previous implementation
# (including main() function, utility functions, etc.)

def main():
    # Simulated historical trading data with more detailed market conditions
    historical_data = [
        {
            'stock_price': 100.0,
            'fundamental_score': 0.7,
            'technical_score': 0.6,
            'sentiment_score': 0.5
        },
        {
            'stock_price': 105.0,
            'fundamental_score': 0.8,
            'technical_score': 0.7,
            'sentiment_score': 0.6
        },
        # Add more historical data points to simulate longer trading periods
        {
            'stock_price': 103.0,
            'fundamental_score': 0.6,
            'technical_score': 0.5,
            'sentiment_score': 0.4
        },
        {
            'stock_price': 108.0,
            'fundamental_score': 0.9,
            'technical_score': 0.8,
            'sentiment_score': 0.7
        }
    ]
    
    # Neural network configuration
    input_size = 6  # stock_price, fundamental_score, technical_score, sentiment_score, current_cash, current_stocks
    hidden_layers = [10, 8, 6]  # Example hidden layer configuration
    output_size = 6  # 3 action probabilities + 3 quantity probabilities
    
    # Create and run genetic algorithm with more generations and logging
    genetic_algo = GeneticTradingAlgorithm(
        input_size=input_size, 
        hidden_layers=hidden_layers, 
        output_size=output_size,
        population_size=20  # Reduced for demonstration
    )
    
    # Evolve best trading neural network with more detailed logging
    best_network = genetic_algo.evolve_generation(
        historical_data, 
        generations=10,  # Reduced for demonstration
        log_top_performers=3
    )
    
    # Print final best network's complete transaction log
    print("\n\n===== FINAL BEST NETWORK FULL TRANSACTION LOG =====")
    best_network.transaction_log.print_log()

if __name__ == "__main__":
    main()


Generation 1:
Best Fitness = 23.023632212406596
Average Fitness = 6.892407317537601

Top Performers Transaction Summaries:

Top Network 1 (ID: network_0):

Transaction Log for Network network_0:
2024-12-04 08:56:04 | Action: buy | Stock Price: $100.00 | Quantity: 1.68 | Cash (Before/After): $1000.00/$832.46 | Stocks (Before/After): 0.00/1.68
2024-12-04 08:56:04 | Action: buy | Stock Price: $105.00 | Quantity: 1.33 | Cash (Before/After): $832.46/$693.00 | Stocks (Before/After): 1.68/3.00
2024-12-04 08:56:04 | Action: buy | Stock Price: $103.00 | Quantity: 1.13 | Cash (Before/After): $693.00/$576.90 | Stocks (Before/After): 3.00/4.13
2024-12-04 08:56:04 | Action: buy | Stock Price: $108.00 | Quantity: 0.89 | Cash (Before/After): $576.90/$480.24 | Stocks (Before/After): 4.13/5.03

Performance Summary:
total_transactions: 4
buy_transactions: 4
sell_transactions: 0
total_stocks_traded: 5.025728502054619
first_transaction: 2024-12-04 08:56:04
last_transaction: 2024-12-04 08:56:04

Top Netwo